In [1]:
import pandas as pd
import pandas_ta as ta
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import random 
import tensorflow as tf
from math import sqrt

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import GRU, Dropout, Dense, RepeatVector, TimeDistributed
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
from keras.losses import Huber 
from keras.regularizers import l2

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv("/Users/alexzheng/Developer/GitHub/EC331-project/data/Bitcoin_data.csv", 
                   index_col=0, 
                   parse_dates=True) # set index to datetime 
data.head()

,open,high,low,close
date,,,,
2018-05-15 06:00:00,8733.86,8796.68,8707.28,8740.99
2018-05-15 07:00:00,8740.99,8766.00,8721.11,8739.00
2018-05-15 08:00:00,8739.00,8750.27,8660.53,8728.49
2018-05-15 09:00:00,8728.49,8754.40,8701.35,8708.32
2018-05-15 10:00:00,8708.32,8865.00,8695.11,8795.90


In [3]:
data.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
open,58146.0,29058.953961,22296.323922,3139.76,9169.0800,23867.00,43658.6750,108293.0
high,58146.0,29185.783902,22388.672956,3158.34,9196.0150,23974.26,43833.7500,108364.0
low,58146.0,28924.437684,22198.723557,3122.28,9138.0000,23761.40,43475.8700,107128.0
close,58146.0,29060.513187,22297.945107,3139.76,9169.0825,23872.00,43661.8725,108276.0


In [4]:
print(f"Missing values:\n{data.isna().sum()}")

Missing values:
open     0
high     0
low      0
close    0
dtype: int64


## Data processing

In [5]:
# !pip install pandas-ta

In [6]:
# Feature engineering 
data['returns'] = data['close'].pct_change() # hourly returns 
data['SMA'] = data['close'].rolling(24).mean() # 24h moving average 
data['RSI'] = ta.rsi(data['close'], timeperiod=14) # relative strength index 

In [7]:
# Handling missing values 
print(f"Missing values:\n{data.isna().sum()}")
data = data.dropna()
print(f"Missing values:\n{data.isna().sum()}")

Missing values:
open        0
high        0
low         0
close       0
returns     1
SMA        23
RSI        14
dtype: int64
Missing values:
open       0
high       0
low        0
close      0
returns    0
SMA        0
RSI        0
dtype: int64


In [8]:
# Temporal split (avoid look-ahead bias)
train_size = int(len(data)*0.7)
val_size = int(len(data)*0.15)
train = data.iloc[:train_size]
val = data.iloc[train_size:train_size+val_size]
test = data.iloc[train_size+val_size:]

print(f"Train size: {train.shape}")
print(f"Val size: {val.shape}")
print(f"Test size: {test.shape}")

Train size: (40686, 7)
Val size: (8718, 7)
Test size: (8719, 7)


In [9]:
# Robust scaling (handles outliers better than MinMax)
scaler = RobustScaler()
train_scaled = scaler.fit_transform(train[['close', 'returns', 'RSI']]) # Use the same scaler for train, val, test sets

In [10]:
data.head()

,open,high,low,close,returns,SMA,RSI
date,,,,,,,
2018-05-16 05:00:00,8229.38,8248.63,8166.25,8220.40,-0.001091,8527.641250,24.936611
2018-05-16 06:00:00,8220.40,8267.12,8176.91,8238.80,0.002238,8506.716667,27.443645
2018-05-16 07:00:00,8238.80,8255.00,8213.79,8232.99,-0.000705,8485.632917,27.135460
2018-05-16 08:00:00,8232.99,8261.24,8200.00,8207.48,-0.003099,8463.924167,25.767236
2018-05-16 09:00:00,8207.48,8263.93,8207.48,8243.45,0.004383,8444.554583,31.046699


In [11]:
# Sequence generation (24h input -> 240h output)
def create_sequence(data, input_steps=24, output_steps=240):
    X, y = [], []
    for i in range(len(data) - input_steps - output_steps):
        X.append(data[i : i+input_steps])
        y.append(data[i + input_steps : i + input_steps + output_steps, 0]) # close price index=0 in 'train_scaled'
    return np.array(X), np.array(y)

X_train, y_train = create_sequence(train_scaled)
X_val, y_val = create_sequence(train_scaled)
X_test, y_test = create_sequence(train_scaled)

### Model architecture 

In [12]:
gru_model = Sequential([
    # Encoder (collapses sequence to vector)
    GRU(64,  # Reduced from 128
        input_shape=(24, 3),
        kernel_regularizer=l2(1e-4),  # Slightly stronger regularization
        dropout=0.5, 
        return_sequences=False  # Outputs 2D: (batch_size, 64)
    ),
    
    # Prepare for sequence generation
    RepeatVector(240),  # Now compatible with 2D encoder output
    
    # Decoder (rebuilds sequence)
    GRU(32,  # Reduced from 64
        kernel_regularizer=l2(5e-5),
        dropout=0.5,
        return_sequences=True
    ),
    
    TimeDistributed(Dense(1))
])

gru_model.compile(
    optimizer=Adam(learning_rate=1e-3),  # Reduced from 5e-3
    loss=Huber()
)
gru_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        13,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 240, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 240, 32)        │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 240, 1)         │            33 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,689 (88.63 KB)

 Trainable params: 22,689 (88.63 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Training 
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = gru_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=256,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
632/632 ━━━━━━━━━━━━━━━━━━━━ 76s 114ms/step - loss: 0.0756 - val_loss: 0.0271
Epoch 2/50
632/632 ━━━━━━━━━━━━━━━━━━━━ 69s 109ms/step - loss: 0.0666 - val_loss: 0.0354
Epoch 3/50
632/632 ━━━━━━━━━━━━━━━━━━━━ 72s 113ms/step - loss: 0.0606 - val_loss: 0.0275
Epoch 4/50
632/632 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - loss: 0.0586

KeyboardInterrupt: 

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss', linestyle='dashed')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

### Evaluation

In [ ]:
# Make predictions with X_val
y_val_pred_scaled = gru_model.predict(X_val)

In [16]:
# Define evaluation function 
def evaluate(y_true, y_pred):
    # Remove last dimension if needed
    if y_pred.ndim == 3:
        y_pred = y_pred.squeeze(-1)
        
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

In [ ]:
y_val_pred_scaled.shape

In [ ]:
y_val.shape

In [ ]:
# Evaluate with the validation set 
mae_val, rmse_val = evaluate(y_val, y_val_pred_scaled)
print(f"MAE: {round(mae_val,4)}, RMSE: {round(rmse_val,4)}")

In [ ]:
# Make predictions with X_test
y_test_pred_scaled = gru_model.predict(X_test)

In [ ]:
# Evaluate with the test set 
mae_test, rmse_test = evaluate(y_test, y_test_pred_scaled)
print(f"MAE: {round(mae_test,4)}, RMSE: {round(rmse_test,4)}")

### Visualizing predictions on the validation set for a sample sequence

In [ ]:
# Choose a sample index (here, the first sequence)
sample_idx_val = 1

# Extract the corresponding date range from the validation DataFrame.
# In the create_sequence function, for i=0 the output (y) corresponds to val.index[24:24+240]
val_dates = val.index[24:24+240]

plt.figure(figsize=(12, 6))
plt.plot(val_dates, y_val[sample_idx_val], label='Actual Price')
plt.plot(val_dates, y_val_pred_scaled[sample_idx_val].squeeze(), 
         label='Predicted Price', linestyle='--')
plt.xlabel('Date')
plt.ylabel('Scaled Close Price')
plt.title('GRU Model Predictions on Validation Set')
plt.legend()
plt.grid(True)
plt.show()

### Visualising predictions on the test set for a sample sequence

In [ ]:
# Choose a sample index (again, the first sequence for illustration)
sample_idx_test = 1

# For the test set, the corresponding dates run from test.index[24] to test.index[24+240]
test_dates = test.index[24:24+240]

plt.figure(figsize=(12, 6))
plt.plot(test_dates, y_test[sample_idx_test], label='Actual Price')
plt.plot(test_dates, y_test_pred_scaled[sample_idx_test].squeeze(), 
         label='Predicted Price', linestyle='--')
plt.xlabel('Date')
plt.ylabel('Scaled Close Price')
plt.title('GRU Model Predictions on Test Set')
plt.legend()
plt.grid(True)
plt.show()